# Projeto final: previsão de churn

## Fase 1. Análise exploratória (parte 1)

Carregamento da base, tamanho, tipos de dados e sumário estatístico.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../datasets/ecommerce_dataset.csv")

df = pd.read_csv(DATA_PATH)
df.head()

### Tamanho da base

In [ ]:
n_linhas, n_colunas = df.shape
print(f"Linhas: {n_linhas}")
print(f"Colunas: {n_colunas}")

### Tipos de dados

In [ ]:
df.info()

In [ ]:
df.dtypes

### Sumário estatístico

In [ ]:
df.describe()

In [ ]:
df.describe(include=["object", "string"])

## Fase 1. Análise exploratória (parte 2)

Nesta parte, vamos medir o desbalanceamento do alvo, observar a distribuição do tempo de relacionamento e calcular as correlações de Pearson entre as variáveis numéricas.

### Distribuição da variável-alvo

In [ ]:
distribuicao_churn = pd.DataFrame(
    {
        "clientes": df["Churn"].value_counts().sort_index(),
        "percentual": df["Churn"].value_counts(normalize=True).sort_index().mul(100),
    }
)
distribuicao_churn.index = ["Permaneceu (0)", "Saiu (1)"]
distribuicao_churn.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(
    data=df,
    x="Churn",
    hue="Churn",
    palette={0: "#4C78A8", 1: "#E45756"},
    legend=False,
    ax=ax,
)
ax.bar_label(ax.containers[0])
ax.bar_label(ax.containers[1])
ax.set(
    title="Distribuição da variável-alvo",
    xlabel="Churn (0 = permaneceu, 1 = saiu)",
    ylabel="Clientes",
)
plt.show()

### Tempo de relacionamento por classe

In [ ]:
df.groupby("Churn")["Tenure"].agg(["count", "mean", "median"]).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(
    data=df,
    x="Tenure",
    hue="Churn",
    bins=30,
    kde=True,
    multiple="layer",
    alpha=0.45,
    palette={0: "#4C78A8", 1: "#E45756"},
    ax=ax,
)
ax.set(
    title="Distribuição do tempo de relacionamento por classe",
    xlabel="Tempo de relacionamento",
    ylabel="Clientes",
)
plt.show()

### Correlação entre variáveis numéricas

`CustomerID` fica fora do cálculo porque é apenas um identificador.

In [ ]:
variaveis_numericas = df.select_dtypes(include="number").drop(columns="CustomerID")
correlacao = variaveis_numericas.corr(method="pearson")

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    correlacao,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Correlação de Pearson entre as variáveis numéricas")
plt.tight_layout()
plt.show()

In [ ]:
(
    correlacao["Churn"]
    .drop("Churn")
    .sort_values(key=lambda valores: valores.abs(), ascending=False)
    .head(5)
)

### Leitura inicial

A base está desbalanceada: só 16,84% dos clientes saíram. A acurácia sozinha pode passar uma impressão errada.

`Tenure` é o sinal mais forte até agora, com correlação de -0,349. Quem saiu tem mediana 1; quem ficou, 10. Reclamação também puxa o churn, com correlação de 0,250, mas nenhum desses fatores sozinho fecha o caso.

Sete colunas numéricas têm valores ausentes, inclusive `Tenure`. Isso precisa ser resolvido antes da engenharia de atributos e da modelagem. O split vai manter a proporção das classes, e o balanceamento ficará só no treino.

# Fase 2. Limpeza e tratamento

A limpeza começa pelas duplicatas e pelos valores ausentes. Depois, usamos boxplots e o intervalo interquartil para entender os extremos antes de decidir se vale alterar algum registro.

## Registros duplicados

In [ ]:
quantidade_duplicadas = int(df.duplicated().sum())
print(f"Linhas duplicadas: {quantidade_duplicadas}")

df_limpo = df.drop_duplicates().copy()
print(f"Formato após a remoção: {df_limpo.shape}")

A busca não encontrou duplicatas. Ainda assim, mantemos `drop_duplicates()` no fluxo para que a limpeza continue válida se a fonte mudar.

## Valores ausentes

Vamos comparar quantidade de nulos, assimetria, média e mediana. Quanto mais distante de zero estiver a assimetria, menos a média representa o centro da distribuição.

In [ ]:
nulos_por_coluna = df_limpo.isna().sum()
colunas_com_nulos = nulos_por_coluna[nulos_por_coluna > 0].index.tolist()

resumo_nulos = pd.DataFrame(
    {
        "ausentes": nulos_por_coluna[colunas_com_nulos],
        "percentual": nulos_por_coluna[colunas_com_nulos].div(len(df_limpo)).mul(100),
        "assimetria": df_limpo[colunas_com_nulos].skew(),
        "media": df_limpo[colunas_com_nulos].mean(),
        "mediana": df_limpo[colunas_com_nulos].median(),
    }
).sort_values("ausentes", ascending=False)

resumo_nulos.round(2)

`HourSpendOnApp` é praticamente simétrica, com assimetria de -0,03, então vamos preencher seus nulos com a média. As outras seis colunas têm cauda à direita e valores extremos. Nelas, a mediana é uma escolha mais segura e muda menos com esses extremos.

In [ ]:
estrategias_imputacao = {
    "Tenure": "mediana",
    "WarehouseToHome": "mediana",
    "HourSpendOnApp": "media",
    "OrderAmountHikeFromlastYear": "mediana",
    "CouponUsed": "mediana",
    "OrderCount": "mediana",
    "DaySinceLastOrder": "mediana",
}

registro_imputacao = []

for coluna, estrategia in estrategias_imputacao.items():
    if estrategia == "media":
        valor = df_limpo[coluna].mean()
    else:
        valor = df_limpo[coluna].median()

    df_limpo[coluna] = df_limpo[coluna].fillna(valor)
    registro_imputacao.append(
        {"coluna": coluna, "estrategia": estrategia, "valor": valor}
    )

pd.DataFrame(registro_imputacao).round({"valor": 2})

In [ ]:
print(f"Valores ausentes após a imputação: {int(df_limpo.isna().sum().sum())}")

## Outliers

O boxplot ajuda a localizar valores distantes do miolo da distribuição. Para contar esses pontos, usamos a regra de 1,5 vezes o intervalo interquartil.

In [ ]:
colunas_para_boxplot = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
    "CashbackAmount",
]

fig, eixos = plt.subplots(2, 4, figsize=(18, 8))

for coluna, eixo in zip(colunas_para_boxplot, eixos.flat):
    sns.boxplot(x=df_limpo[coluna], color="#4C78A8", ax=eixo)
    eixo.set_title(coluna)
    eixo.set_xlabel("")

fig.suptitle("Distribuição das variáveis numéricas", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
registro_outliers = []

for coluna in colunas_para_boxplot:
    primeiro_quartil = df_limpo[coluna].quantile(0.25)
    terceiro_quartil = df_limpo[coluna].quantile(0.75)
    intervalo_interquartil = terceiro_quartil - primeiro_quartil
    limite_inferior = primeiro_quartil - 1.5 * intervalo_interquartil
    limite_superior = terceiro_quartil + 1.5 * intervalo_interquartil
    fora_dos_limites = (
        (df_limpo[coluna] < limite_inferior)
        | (df_limpo[coluna] > limite_superior)
    )
    quantidade = int(fora_dos_limites.sum())

    registro_outliers.append(
        {
            "coluna": coluna,
            "limite_inferior": limite_inferior,
            "limite_superior": limite_superior,
            "outliers": quantidade,
            "percentual": quantidade / len(df_limpo) * 100,
        }
    )

resumo_outliers = pd.DataFrame(registro_outliers).sort_values(
    "percentual", ascending=False
)
resumo_outliers.round(2)

### Decisão sobre os outliers

Vamos manter os valores. A regra do IQR marcou 12,49% de `OrderCount` e 11,17% de `CouponUsed`, mas são contagens plausíveis de clientes mais ativos. Cortá-las apagaria justamente parte do comportamento que queremos estudar. Os extremos das outras colunas também não bastam, sozinhos, para afirmar que houve erro de cadastro.

Essa escolha exige cuidado com o KNN, porque distâncias grandes pesam no resultado. Na etapa de modelagem, o escalonamento será ajustado apenas no treino e vamos comparar as métricas de treino e teste. A Árvore de Decisão é menos sensível a esses extremos.

## Resultado da limpeza

Não havia duplicatas para remover. Os nulos foram preenchidos com média em `HourSpendOnApp` e mediana nas outras seis colunas. Os outliers foram mantidos porque parecem representar comportamento real, não erro confirmado.

In [ ]:
pd.Series(
    {
        "linhas": len(df_limpo),
        "colunas": df_limpo.shape[1],
        "duplicatas": int(df_limpo.duplicated().sum()),
        "valores_ausentes": int(df_limpo.isna().sum().sum()),
    },
    name="resultado",
)